# 09 — Fine-tune Qwen2-VL (LoRA) on EcoExplain train

**Run on a GPU machine** (friend's PC, Colab, or Ada). Keeps notebooks **05–08** as zero-shot baselines; this adds **`qwen_vl_finetuned_ecoexplain`**.

- **Train:** `train` split only (256 patches) — never use `test` for training.
- **Eval:** `test` split (55 patches), same metric as 05–08.
- **Needs:** `data/processed_images/*.jpg`, `dataset.jsonl`, `data/splits/`, CUDA GPU recommended.

CLI equivalent: `python scripts/run_finetune_qwen.py --train --eval --merge`


## 1. Environment (GPU + packages)


In [ ]:
from pathlib import Path
import json
import sys

import torch

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

print("python", sys.executable)
print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))
else:
    print("WARNING: no CUDA — training will be very slow on CPU/MPS")

# Install once on the GPU machine:
# pip install torch torchvision transformers accelerate peft bitsandbytes


## 2. Config


In [ ]:
from src.vlm_finetune import FinetuneConfig, FINETUNE_COMBO_ID

JSONL = ROOT / "dataset.jsonl"
SPLITS = ROOT / "data" / "splits"
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)

# Use 2B on one consumer GPU; switch to Qwen/Qwen2-VL-7B-Instruct only if VRAM allows.
cfg = FinetuneConfig(
    base_model_id="Qwen/Qwen2-VL-2B-Instruct",
    adapter_dir="results/checkpoints/qwen2vl_lora_ecoexplain",
    epochs=3,
    learning_rate=2e-4,
    max_train_samples=None,   # None = all 256 train patches; use 32 for a quick smoke test
    use_4bit=True,            # NVIDIA + bitsandbytes; set False if import fails
)

SPLIT_EVAL = "test"
MAX_EVAL = 55
DO_TRAIN = True
DO_EVAL = True

print("combo_id", FINETUNE_COMBO_ID)
print("adapter", ROOT / cfg.adapter_dir)


## 3. Train LoRA (train split only)


In [ ]:
from src.vlm_finetune import train_lora

if DO_TRAIN:
    adapter_path = train_lora(ROOT, cfg)
    print("adapter saved to", adapter_path)
else:
    print("skipped training (DO_TRAIN=False)")


## 4. Evaluate on test (55 patches)


In [ ]:
from src.vlm_finetune import eval_finetuned
from src.vlm_eval import format_accuracy_pct

if DO_EVAL:
    report = eval_finetuned(ROOT, split=SPLIT_EVAL, max_eval=MAX_EVAL, cfg=cfg)
    print("accuracy", format_accuracy_pct(report.get("accuracy")))
    print("wrote", RESULTS / f"vlm_combo_{FINETUNE_COMBO_ID}.json")


## 5. Scorecard


In [ ]:
from src.vlm_eval import format_accuracy_pct

path = RESULTS / f"vlm_combo_{FINETUNE_COMBO_ID}.json"
report = json.loads(path.read_text())
import pandas as pd
pd.DataFrame([{
    "combo_id": report["combo_id"],
    "base_model": report.get("hf_model_id"),
    "adapter": report.get("adapter_dir"),
    "status": report["status"],
    "accuracy (%)": format_accuracy_pct(report.get("accuracy")),
    "majority baseline (%)": format_accuracy_pct(report.get("majority_accuracy")),
    "n_eval": report.get("n_eval"),
    "n_parsed": report.get("n_parsed_class"),
    "seconds": report.get("seconds"),
}])


## 6. Compare with zero-shot 05–08 (optional)


In [ ]:
from src.vlm_finetune import merge_with_baselines
from src.vlm_eval import compare_reports

comparison = merge_with_baselines(ROOT)
print("BEST", comparison.get("best_combo_id"), comparison.get("best_accuracy_pct"))
import pandas as pd
tbl = pd.DataFrame(comparison["table"])
show = [c for c in ["combo_id", "family", "status", "accuracy_pct", "n_eval", "seconds"] if c in tbl.columns]
display(tbl[show])
print("wrote", RESULTS / "vlm_combo_comparison_with_finetune.json")
